# Phase A — Data Exploration

This notebook verifies the GRaM point-cloud data layout, split logic, preprocessing, and quick visual sanity checks before any model training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.datasets.gram_dataset import GRAMDataset, scan_dataset_files
from src.datasets.preprocessing import PhaseAPreprocessor, fit_velocity_stats
from src.datasets.splits import build_geometry_split, summarize_split

plt.style.use('seaborn-v0_8')

In [ ]:
config = yaml.safe_load((ROOT / 'configs' / 'phase_a.yaml').read_text())
records = scan_dataset_files(ROOT / config['data']['root'])
split = build_geometry_split(records, val_fraction=config['split']['val_fraction'], seed=config['split']['seed'])
print(summarize_split(split))

train_dataset = GRAMDataset(ROOT / config['data']['root'], records=split['train_records'])
velocity_stats = fit_velocity_stats(train_dataset)
preprocessor = PhaseAPreprocessor(
    coordinate_mode=config['preprocessing']['coordinate_mode'],
    velocity_mean=velocity_stats['mean'],
    velocity_std=velocity_stats['std'],
    subsample_points=config['preprocessing']['subsample_points'],
    seed=config['split']['seed'],
)
sample = preprocessor(train_dataset[0])
sample['metadata']

In [ ]:
plot_idx = np.random.default_rng(0).choice(sample['pos'].shape[0], size=min(5000, sample['pos'].shape[0]), replace=False)
plot_pos = sample['pos'][plot_idx]
plot_airfoil = sample['airfoil_mask'][plot_idx]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(plot_pos[~plot_airfoil, 0], plot_pos[~plot_airfoil, 1], plot_pos[~plot_airfoil, 2], s=1, alpha=0.2, label='fluid')
ax.scatter(plot_pos[plot_airfoil, 0], plot_pos[plot_airfoil, 1], plot_pos[plot_airfoil, 2], s=6, alpha=0.9, label='airfoil')
ax.set_title('Point cloud with airfoil mask')
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sample['boundary_distance'], bins=40)
axes[0].set_title('Boundary-distance distribution')
axes[0].set_xlabel('distance')

vel_in_mag = np.linalg.norm(sample['velocity_in'], axis=-1).mean(axis=1)
vel_out_mag = np.linalg.norm(sample['velocity_out'], axis=-1).mean(axis=1)
axes[1].plot(range(1, 6), vel_in_mag, marker='o', label='input')
axes[1].plot(range(1, 6), vel_out_mag, marker='o', label='target')
axes[1].set_title('Mean velocity magnitude by frame')
axes[1].set_xlabel('frame index')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for record in records[:20]:
    item = GRAMDataset(ROOT / config['data']['root'], records=[record])[0]
    rows.append({
        'file_name': item['metadata']['file_name'],
        'simulation_id': item['metadata']['simulation_id'],
        'num_points': item['metadata']['num_points'],
        'airfoil_fraction': item['metadata']['num_airfoil_points'] / item['metadata']['num_points'],
        'velocity_abs_mean': float(np.abs(item['velocity_in']).mean()),
    })

pd.DataFrame(rows).head()